# V18-C9B: Stratified FDR + C3-Only Gene Rerun
## GSE182159 Tissue-Separated Reanalysis — ITLAS Analytical Workflow

**Date:** 2026-03-04  
**Purpose:** Fix two critical issues from C9 initial run

| Fix | Issue | Approach |
|-----|-------|----------|
| Fix 1B | FDR family too large (0% survival) | Stratified BH: comparison×tissue, comparison×tissue×lineage |
| Fix 3B | Key genes (IFN, SOCS, AICDA, HLA) C3-only | Rerun C5-style analysis for 20 critical C3-only genes |

**Input:** Existing C3/C4/C5/C7 CSVs + h5ad  
**Output:** `C9_method_fixes/Fix1B_StratifiedFDR/` + `C9_method_fixes/Fix3B_C3onlyGenes/`

---
> **Rules:** Liver/Blood separate | dot/box only | Liver=red● Blood=blue▲ | No combined graphs  
> **Original data files are NEVER modified.**


In [1]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 66.3 MB/s eta 0:00:00
  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total


## Cell 1. Setup & Google Drive Mount

In [2]:
# ================================================================
# CELL 1: SETUP
# ================================================================
import os, warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/ITLAS/results/version18-analysis'
DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
OUT_BASE = os.path.join(BASE, 'C9_method_fixes')

# Create output subdirectories
for d in ['Fix1B_StratifiedFDR', 'Fix3B_C3onlyGenes', 'Fix3B_C3onlyGenes/dotplots']:
    os.makedirs(os.path.join(OUT_BASE, d), exist_ok=True)

print("Setup complete")
print(f"  BASE: {BASE}")
print(f"  OUT:  {OUT_BASE}")


Mounted at /content/drive
Setup complete
  BASE: /content/drive/MyDrive/ITLAS/results/version18-analysis
  OUT:  /content/drive/MyDrive/ITLAS/results/version18-analysis/C9_method_fixes


## Cell 2. Load Existing Results

In [3]:
# ================================================================
# CELL 2: LOAD DATA
# ================================================================
import pandas as pd
import numpy as np

# --- C3 statistics ---
c3_path = os.path.join(BASE, 'C3_gene_expression')
c3_files = [f for f in os.listdir(c3_path) if f.endswith('.csv') or f.endswith('.csv.gz')]
print(f"C3 files: {c3_files}")

# Try loading main statistics file
for fname in ['C3_all_statistics.csv.gz', 'C3_all_statistics.csv']:
    fpath = os.path.join(c3_path, fname)
    if os.path.exists(fpath):
        df_c3 = pd.read_csv(fpath)
        print(f"  C3 stats loaded: {df_c3.shape}")
        break

# --- C4 pathway ---
c4_liver = pd.read_csv(os.path.join(BASE, 'C4_pathway/C4_pathway_liver.csv'))
c4_blood = pd.read_csv(os.path.join(BASE, 'C4_pathway/C4_pathway_blood.csv'))
print(f"  C4 Liver: {c4_liver.shape}, Blood: {c4_blood.shape}")

# --- C5 genes ---
c5_liver = pd.read_csv(os.path.join(BASE, 'C5_genes/C5_genes_liver.csv'))
c5_blood = pd.read_csv(os.path.join(BASE, 'C5_genes/C5_genes_blood.csv'))
print(f"  C5 Liver: {c5_liver.shape}, Blood: {c5_blood.shape}")

# --- C7 correlations ---
df_c7 = pd.read_csv(os.path.join(BASE, 'C7_patterns/C7_gene_gene_correlations.csv'))
print(f"  C7 correlations: {df_c7.shape}")

# Detect p-value column names
for df_name, df in [('C3', df_c3), ('C4_Liver', c4_liver), ('C5_Liver', c5_liver), ('C7', df_c7)]:
    pcols = [c for c in df.columns if 'p_val' in c.lower() or c == 'p_value' or c == 'pvalue' or c == 'p']
    print(f"  {df_name} p-value columns: {pcols}")

# Detect comparison/tissue/lineage columns
for df_name, df in [('C3', df_c3), ('C5_Liver', c5_liver)]:
    print(f"  {df_name} columns: {list(df.columns)}")

print("\nData loading complete")


C3 files: ['C3_gene_list_196genes.csv', 'C3_liver_donor_gene_expression.csv.gz', 'C3_blood_donor_gene_expression.csv.gz', 'C3_all_statistics.csv.gz', 'C3_NL_vs_IT_all.csv', 'C3_NL_vs_IT_significant.csv', 'C3_liver_blood_discrepancy_NL_IT.csv', 'C3_pattern_classification.csv']
  C3 stats loaded: (16464, 16)
  C4 Liver: (1274, 15), Blood: (1274, 15)
  C5 Liver: (7252, 17), Blood: (7252, 17)
  C7 correlations: (1470, 8)
  C3 p-value columns: ['p_value']
  C4_Liver p-value columns: ['p_value']
  C5_Liver p-value columns: ['p_value']
  C7 p-value columns: ['p_value']
  C3 columns: ['tissue', 'lineage', 'gene', 'comparison', 'grp1', 'grp2', 'n1', 'n2', 'mean_grp1', 'mean_grp2', 'pct_change', 'direction', 'consistency', 'U_stat', 'p_value', 'sig']
  C5_Liver columns: ['tissue', 'gene', 'lineage', 'pathway', 'comparison', 'stage1', 'stage2', 'mean_s1', 'mean_s2', 'pct_change', 'direction', 'p_value', 'sig', 'consistency', 'consist_pct', 'n_s1', 'n_s2']

Data loading complete


## Cell 3. Fix 1B — Stratified FDR Correction

**Problem:** C9 initial run applied BH-FDR to ALL tests in one family (8,232 for C3).  
With n=6 vs n=6, minimum achievable p=0.0022 → impossible to survive FDR at family=8,232.

**Solution:** Apply FDR at three stratification levels:
1. **Level A:** `comparison × tissue` (e.g., NL→IT Liver = ~1,176 tests for C3)
2. **Level B:** `comparison × tissue × lineage` (e.g., NL→IT Liver Myeloid = ~196 tests for C3)
3. **Level C:** Per-comparison only (tissue combined) for reference

Report survival rates at each level to determine optimal reporting strategy.


In [4]:
# ================================================================
# CELL 3: FIX 1B — STRATIFIED FDR
# ================================================================
from scipy.stats import false_discovery_control  # scipy >= 1.11
# Fallback for older scipy:
try:
    from scipy.stats import false_discovery_control as fdc
    USE_SCIPY_FDC = True
except ImportError:
    USE_SCIPY_FDC = False

from statsmodels.stats.multitest import multipletests

def apply_bh_fdr(pvals):
    """Apply Benjamini-Hochberg FDR correction. Returns q-values."""
    mask = ~np.isnan(pvals)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    if mask.sum() > 0:
        _, qv, _, _ = multipletests(pvals[mask], alpha=0.05, method='fdr_bh')
        qvals[mask] = qv
    return qvals

def stratified_fdr(df, p_col, group_cols, label=""):
    """Apply BH-FDR within each stratum defined by group_cols."""
    df = df.copy()
    q_col = f'q_value_{label}'
    sig_col = f'fdr_sig_{label}'
    df[q_col] = np.nan
    df[sig_col] = False

    for name, grp in df.groupby(group_cols):
        idx = grp.index
        pvals = grp[p_col].values.astype(float)
        qvals = apply_bh_fdr(pvals)
        df.loc[idx, q_col] = qvals
        df.loc[idx, sig_col] = qvals < 0.05

    return df

# ── Detect column names ──
# C3
p_col_c3 = [c for c in df_c3.columns if 'p_val' in c.lower() or c == 'p_value'][0]
tissue_col_c3 = 'tissue' if 'tissue' in df_c3.columns else [c for c in df_c3.columns if 'tissue' in c.lower()][0]
comp_col_c3 = [c for c in df_c3.columns if 'compar' in c.lower()][0] if any('compar' in c.lower() for c in df_c3.columns) else None
lineage_col_c3 = 'lineage' if 'lineage' in df_c3.columns else [c for c in df_c3.columns if 'lineage' in c.lower()][0]

# If no explicit comparison column, construct from grp1/grp2
if comp_col_c3 is None:
    grp_cols = [c for c in df_c3.columns if 'grp' in c.lower()]
    if len(grp_cols) >= 2:
        df_c3['comparison'] = df_c3[grp_cols[0]].astype(str) + '_vs_' + df_c3[grp_cols[1]].astype(str)
        comp_col_c3 = 'comparison'
    else:
        print("WARNING: Cannot find comparison column in C3. Using 'comparison' if present.")
        comp_col_c3 = 'comparison'

print(f"C3: p={p_col_c3}, tissue={tissue_col_c3}, comp={comp_col_c3}, lineage={lineage_col_c3}")
print(f"C3 comparisons: {df_c3[comp_col_c3].unique()}")
print(f"C3 tissues: {df_c3[tissue_col_c3].unique()}")
print(f"C3 lineages: {df_c3[lineage_col_c3].unique()}")
print()

# ── Apply 3-level stratified FDR to C3 ──
print("=" * 60)
print("  FIX 1B: STRATIFIED FDR — C3 (196 genes)")
print("=" * 60)

# Level A: comparison × tissue
df_c3 = stratified_fdr(df_c3, p_col_c3, [comp_col_c3, tissue_col_c3], label='compXtissue')
# Level B: comparison × tissue × lineage
df_c3 = stratified_fdr(df_c3, p_col_c3, [comp_col_c3, tissue_col_c3, lineage_col_c3], label='compXtissueXlineage')
# Level C: comparison only
df_c3 = stratified_fdr(df_c3, p_col_c3, [comp_col_c3], label='compOnly')

# Report
nominal = (df_c3[p_col_c3] < 0.05).sum()
for level, col in [('A: comp×tissue', 'fdr_sig_compXtissue'),
                    ('B: comp×tissue×lineage', 'fdr_sig_compXtissueXlineage'),
                    ('C: comp only', 'fdr_sig_compOnly')]:
    surv = df_c3[col].sum()
    rate = surv / nominal * 100 if nominal > 0 else 0
    print(f"  Level {level}: {nominal} nominal → {surv} FDR(q<0.05) [{rate:.1f}%]")

# Per-tissue breakdown for Level B
print()
for tissue in df_c3[tissue_col_c3].unique():
    mask = df_c3[tissue_col_c3] == tissue
    nom = (df_c3.loc[mask, p_col_c3] < 0.05).sum()
    surv_a = df_c3.loc[mask, 'fdr_sig_compXtissue'].sum()
    surv_b = df_c3.loc[mask, 'fdr_sig_compXtissueXlineage'].sum()
    print(f"  {tissue}: nominal={nom} → Level A={surv_a}, Level B={surv_b}")

# NL→IT specific breakdown
print()
print("── NL→IT Specific ──")
for tissue in df_c3[tissue_col_c3].unique():
    # Find NL→IT comparison string
    comps = df_c3[comp_col_c3].unique()
    it_comp = [c for c in comps if ('NL' in str(c) and 'IT' in str(c)) or ('Normal' in str(c) and 'IT' in str(c))]
    if not it_comp:
        it_comp = [c for c in comps if 'IT' in str(c)]
    if it_comp:
        mask = (df_c3[comp_col_c3] == it_comp[0]) & (df_c3[tissue_col_c3] == tissue)
        nom = (df_c3.loc[mask, p_col_c3] < 0.05).sum()
        surv_b = df_c3.loc[mask, 'fdr_sig_compXtissueXlineage'].sum()
        print(f"  NL→IT {tissue}: nominal={nom} → Level B={surv_b}")

        # Show surviving genes
        if surv_b > 0:
            survivors = df_c3.loc[mask & df_c3['fdr_sig_compXtissueXlineage']]
            for _, row in survivors.iterrows():
                gene = row.get('gene', row.get('Gene', '?'))
                lin = row[lineage_col_c3]
                pv = row[p_col_c3]
                qv = row['q_value_compXtissueXlineage']
                print(f"    ★ {gene}/{lin}: p={pv:.4f}, q={qv:.4f}")

# Save C3
out_c3 = os.path.join(OUT_BASE, 'Fix1B_StratifiedFDR', 'C3_stratified_FDR.csv')
df_c3.to_csv(out_c3, index=False)
print(f"\nSaved: {out_c3}")


C3: p=p_value, tissue=tissue, comp=comparison, lineage=lineage
C3 comparisons: ['NL→IT' 'NL→IA' 'NL→AR' 'NL→CR' 'IT→IA' 'IA→AR' 'CR→AR']
C3 tissues: ['Liver' 'Blood']
C3 lineages: ['Myeloid' 'CD4_T' 'CD8_T' 'NK' 'B' 'PlasmaB']

  FIX 1B: STRATIFIED FDR — C3 (196 genes)
  Level A: comp×tissue: 1442 nominal → 0 FDR(q<0.05) [0.0%]
  Level B: comp×tissue×lineage: 1442 nominal → 53 FDR(q<0.05) [3.7%]
  Level C: comp only: 1442 nominal → 0 FDR(q<0.05) [0.0%]

  Liver: nominal=583 → Level A=0, Level B=0
  Blood: nominal=859 → Level A=0, Level B=53

── NL→IT Specific ──
  NL→IT Liver: nominal=139 → Level B=0
  NL→IT Blood: nominal=253 → Level B=53
    ★ AIM2/Myeloid: p=0.0054, q=0.0278
    ★ AKT1/Myeloid: p=0.0051, q=0.0278
    ★ ATR/Myeloid: p=0.0025, q=0.0198
    ★ BACH2/Myeloid: p=0.0082, q=0.0374
    ★ BAK1/Myeloid: p=0.0025, q=0.0198
    ★ BBC3/Myeloid: p=0.0057, q=0.0278
    ★ CASP4/Myeloid: p=0.0051, q=0.0278
    ★ CASP8/Myeloid: p=0.0101, q=0.0374
    ★ CD74/Myeloid: p=0.0025, q=0.0198

## Cell 4. Fix 1B — Stratified FDR for C4 Pathways

In [6]:
# ================================================================
# CELL 4: FIX 1B — C4 STRATIFIED FDR
# ================================================================
print("=" * 60)
print("  FIX 1B: STRATIFIED FDR — C4 (Pathways)")
print("=" * 60)

for tissue_label, df_c4 in [('Liver', c4_liver.copy()), ('Blood', c4_blood.copy())]:
    # Detect columns
    p_col = [c for c in df_c4.columns if 'p_val' in c.lower() or c == 'p_value' or c == 'pvalue'][0]
    comp_col = [c for c in df_c4.columns if 'compar' in c.lower()][0] if any('compar' in c.lower() for c in df_c4.columns) else None
    lineage_col = [c for c in df_c4.columns if 'lineage' in c.lower()][0] if any('lineage' in c.lower() for c in df_c4.columns) else None

    if comp_col is None:
        grp_cols = [c for c in df_c4.columns if 'grp' in c.lower()]
        if len(grp_cols) >= 2:
            df_c4['comparison'] = df_c4[grp_cols[0]].astype(str) + '_vs_' + df_c4[grp_cols[1]].astype(str)
            comp_col = 'comparison'

    print(f"\n  {tissue_label}: p={p_col}, comp={comp_col}, lineage={lineage_col}")

    # Level A: comparison only (since tissue is already separated)
    df_c4 = stratified_fdr(df_c4, p_col, [comp_col], label='compOnly')

    # Level B: comparison × lineage
    if lineage_col:
        df_c4 = stratified_fdr(df_c4, p_col, [comp_col, lineage_col], label='compXlineage')

    nominal = (df_c4[p_col] < 0.05).sum()
    surv_a = df_c4['fdr_sig_compOnly'].sum()
    surv_b = df_c4['fdr_sig_compXlineage'].sum() if lineage_col else 0
    print(f"  {tissue_label}: nominal={nominal} → comp-only={surv_a}, comp×lineage={surv_b}")

    # NL→IT breakdown
    comps = df_c4[comp_col].unique()
    it_comp = [c for c in comps if ('NL' in str(c) and 'IT' in str(c))]
    if not it_comp:
        it_comp = [c for c in comps if 'IT' in str(c)]
    if it_comp and lineage_col:
        mask_it = df_c4[comp_col] == it_comp[0]
        nom_it = (df_c4.loc[mask_it, p_col] < 0.05).sum()
        surv_it = df_c4.loc[mask_it, 'fdr_sig_compXlineage'].sum()
        print(f"  NL→IT {tissue_label}: nominal={nom_it} → comp×lineage={surv_it}")
        if surv_it > 0:
            survivors = df_c4.loc[mask_it & df_c4['fdr_sig_compXlineage']]
            pw_col = [c for c in df_c4.columns if 'pathway' in c.lower() or 'geneset' in c.lower()]
            pw_col = pw_col[0] if pw_col else 'pathway'
            for _, row in survivors.iterrows():
                pw = row.get(pw_col, '?') if pw_col in row.index else '?'
                lin = row[lineage_col]
                pv = row[p_col]
                qv = row['q_value_compXlineage']
                print(f"    ★ {pw}/{lin}: p={pv:.4f}, q={qv:.4f}")

    # Save
    out_path = os.path.join(OUT_BASE, 'Fix1B_StratifiedFDR', f'C4_{tissue_label}_stratified_FDR.csv')
    df_c4.to_csv(out_path, index=False)
    print(f"  Saved: {out_path}")


  FIX 1B: STRATIFIED FDR — C4 (Pathways)

  Liver: p=p_value, comp=comparison, lineage=lineage
  Liver: nominal=108 → comp-only=0, comp×lineage=0
  NL→IT Liver: nominal=28 → comp×lineage=0
  Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis/C9_method_fixes/Fix1B_StratifiedFDR/C4_Liver_stratified_FDR.csv

  Blood: p=p_value, comp=comparison, lineage=lineage
  Blood: nominal=130 → comp-only=0, comp×lineage=20
  NL→IT Blood: nominal=39 → comp×lineage=11
    ★ exhaustion/B: p=0.0051, q=0.0263
    ★ memory_t/B: p=0.0101, q=0.0438
    ★ stemness/B: p=0.0051, q=0.0263
    ★ mito_dysfunction/B: p=0.0051, q=0.0263
    ★ apoptosis/B: p=0.0051, q=0.0263
    ★ cell_cycle/B: p=0.0051, q=0.0263
    ★ inflammasome/Myeloid: p=0.0061, q=0.0315
    ★ il15_mtor/Myeloid: p=0.0061, q=0.0315
    ★ immune_evasion/Myeloid: p=0.0061, q=0.0315
    ★ mito_dysfunction/Myeloid: p=0.0061, q=0.0315
    ★ epigenetics/Myeloid: p=0.0061, q=0.0315
  Saved: /content/drive/MyDrive/ITLAS/results/version18-anal

## Cell 5. Fix 1B — Stratified FDR for C5 Genes

In [7]:
# ================================================================
# CELL 5: FIX 1B — C5 STRATIFIED FDR
# ================================================================
print("=" * 60)
print("  FIX 1B: STRATIFIED FDR — C5 (148 genes)")
print("=" * 60)

for tissue_label, df_c5 in [('Liver', c5_liver.copy()), ('Blood', c5_blood.copy())]:
    p_col = [c for c in df_c5.columns if 'p_val' in c.lower() or c == 'p_value'][0]
    comp_col = [c for c in df_c5.columns if 'compar' in c.lower()][0] if any('compar' in c.lower() for c in df_c5.columns) else None
    lineage_col = [c for c in df_c5.columns if 'lineage' in c.lower()][0] if any('lineage' in c.lower() for c in df_c5.columns) else None

    if comp_col is None:
        grp_cols = [c for c in df_c5.columns if 'grp' in c.lower()]
        if len(grp_cols) >= 2:
            df_c5['comparison'] = df_c5[grp_cols[0]].astype(str) + '_vs_' + df_c5[grp_cols[1]].astype(str)
            comp_col = 'comparison'

    print(f"\n  {tissue_label}: p={p_col}, comp={comp_col}, lineage={lineage_col}")

    # Level A: comparison only
    df_c5 = stratified_fdr(df_c5, p_col, [comp_col], label='compOnly')

    # Level B: comparison × lineage
    if lineage_col:
        df_c5 = stratified_fdr(df_c5, p_col, [comp_col, lineage_col], label='compXlineage')

    nominal = (df_c5[p_col] < 0.05).sum()
    surv_a = df_c5['fdr_sig_compOnly'].sum()
    surv_b = df_c5['fdr_sig_compXlineage'].sum() if lineage_col else 0
    print(f"  {tissue_label}: nominal={nominal} → comp-only={surv_a}, comp×lineage={surv_b}")

    # NL→IT breakdown
    comps = df_c5[comp_col].unique()
    it_comp = [c for c in comps if ('NL' in str(c) and 'IT' in str(c))]
    if not it_comp:
        it_comp = [c for c in comps if 'IT' in str(c)]
    if it_comp and lineage_col:
        mask_it = df_c5[comp_col] == it_comp[0]
        nom_it = (df_c5.loc[mask_it, p_col] < 0.05).sum()
        surv_it = df_c5.loc[mask_it, 'fdr_sig_compXlineage'].sum()
        print(f"  NL→IT {tissue_label}: nominal={nom_it} → comp×lineage={surv_it}")
        if surv_it > 0:
            gene_col = [c for c in df_c5.columns if c.lower() == 'gene'][0]
            survivors = df_c5.loc[mask_it & df_c5['fdr_sig_compXlineage']]
            for _, row in survivors.iterrows():
                g = row[gene_col]
                lin = row[lineage_col]
                pv = row[p_col]
                qv = row['q_value_compXlineage']
                print(f"    ★ {g}/{lin}: p={pv:.4f}, q={qv:.4f}")

    out_path = os.path.join(OUT_BASE, 'Fix1B_StratifiedFDR', f'C5_{tissue_label}_stratified_FDR.csv')
    df_c5.to_csv(out_path, index=False)
    print(f"  Saved: {out_path}")


  FIX 1B: STRATIFIED FDR — C5 (148 genes)

  Liver: p=p_value, comp=comparison, lineage=lineage
  Liver: nominal=398 → comp-only=0, comp×lineage=0
  NL→IT Liver: nominal=87 → comp×lineage=0
  Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis/C9_method_fixes/Fix1B_StratifiedFDR/C5_Liver_stratified_FDR.csv

  Blood: p=p_value, comp=comparison, lineage=lineage
  Blood: nominal=610 → comp-only=0, comp×lineage=0
  NL→IT Blood: nominal=172 → comp×lineage=0
  Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis/C9_method_fixes/Fix1B_StratifiedFDR/C5_Blood_stratified_FDR.csv


## Cell 6. Fix 1B — C7 Correlations (Lineage-Stratified FDR)

In [8]:
# ================================================================
# CELL 6: FIX 1B — C7 STRATIFIED FDR
# ================================================================
print("=" * 60)
print("  FIX 1B: STRATIFIED FDR — C7 (Correlations)")
print("=" * 60)

p_col_c7 = [c for c in df_c7.columns if 'p_val' in c.lower() or c == 'p_value' or c == 'pvalue'][0]
lineage_col_c7 = [c for c in df_c7.columns if 'lineage' in c.lower()][0] if any('lineage' in c.lower() for c in df_c7.columns) else None
tissue_col_c7 = [c for c in df_c7.columns if 'tissue' in c.lower()][0] if any('tissue' in c.lower() for c in df_c7.columns) else None

print(f"  C7: p={p_col_c7}, lineage={lineage_col_c7}, tissue={tissue_col_c7}")

# Global FDR (same as C9 original)
df_c7_out = df_c7.copy()
pvals = df_c7_out[p_col_c7].values.astype(float)
df_c7_out['q_value_global'] = apply_bh_fdr(pvals)
df_c7_out['fdr_sig_global'] = df_c7_out['q_value_global'] < 0.05

# Stratified by lineage-tissue combo
if lineage_col_c7 and tissue_col_c7:
    strat_col = df_c7_out[tissue_col_c7].astype(str) + '_' + df_c7_out[lineage_col_c7].astype(str)
    df_c7_out['lineage_tissue'] = strat_col
    df_c7_out = stratified_fdr(df_c7_out, p_col_c7, ['lineage_tissue'], label='lineageXtissue')
elif lineage_col_c7:
    df_c7_out = stratified_fdr(df_c7_out, p_col_c7, [lineage_col_c7], label='lineageXtissue')

nominal = (df_c7_out[p_col_c7] < 0.05).sum()
surv_g = df_c7_out['fdr_sig_global'].sum()
surv_s = df_c7_out['fdr_sig_lineageXtissue'].sum() if 'fdr_sig_lineageXtissue' in df_c7_out.columns else 0
print(f"  C7: nominal={nominal} → global={surv_g}, lineage×tissue={surv_s}")

out_path = os.path.join(OUT_BASE, 'Fix1B_StratifiedFDR', 'C7_stratified_FDR.csv')
df_c7_out.to_csv(out_path, index=False)
print(f"  Saved: {out_path}")


  FIX 1B: STRATIFIED FDR — C7 (Correlations)
  C7: p=p_value, lineage=lineage, tissue=tissue
  C7: nominal=517 → global=308, lineage×tissue=333
  Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis/C9_method_fixes/Fix1B_StratifiedFDR/C7_stratified_FDR.csv


## Cell 7. Fix 1B — Comprehensive FDR Summary

In [9]:
# ================================================================
# CELL 7: FIX 1B — SUMMARY TABLE
# ================================================================
print("=" * 60)
print("  FIX 1B: STRATIFIED FDR SUMMARY")
print("=" * 60)
print()
print(f"{'Dataset':<20} {'Nominal':<10} {'Global':<10} {'Comp×Tissue':<15} {'Comp×Tissue×Lin':<18} {'Best Strategy'}")
print("-" * 95)

# Collect from saved CSVs
summary_rows = []

# C3
c3_reload = pd.read_csv(os.path.join(OUT_BASE, 'Fix1B_StratifiedFDR', 'C3_stratified_FDR.csv'))
for tissue in c3_reload[tissue_col_c3].unique():
    m = c3_reload[tissue_col_c3] == tissue
    nom = (c3_reload.loc[m, p_col_c3] < 0.05).sum()
    g_a = c3_reload.loc[m, 'fdr_sig_compXtissue'].sum()
    g_b = c3_reload.loc[m, 'fdr_sig_compXtissueXlineage'].sum()
    best = 'B' if g_b > g_a else 'A' if g_a > 0 else 'nominal+ES'
    row = f"C3_{tissue:<15} {nom:<10} {'—':<10} {g_a:<15} {g_b:<18} {best}"
    print(f"  {row}")
    summary_rows.append({'dataset': f'C3_{tissue}', 'nominal': nom, 'fdr_compXtissue': int(g_a), 'fdr_compXtissueXlineage': int(g_b)})

# C4
for tissue_label in ['Liver', 'Blood']:
    fpath = os.path.join(OUT_BASE, 'Fix1B_StratifiedFDR', f'C4_{tissue_label}_stratified_FDR.csv')
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        p_col = [c for c in df.columns if 'p_val' in c.lower() or c == 'p_value'][0]
        nom = (df[p_col] < 0.05).sum()
        g_a = df['fdr_sig_compOnly'].sum() if 'fdr_sig_compOnly' in df.columns else 0
        g_b = df['fdr_sig_compXlineage'].sum() if 'fdr_sig_compXlineage' in df.columns else 0
        print(f"  C4_{tissue_label:<15} {nom:<10} {'—':<10} {int(g_a):<15} {int(g_b):<18}")
        summary_rows.append({'dataset': f'C4_{tissue_label}', 'nominal': nom, 'fdr_compOnly': int(g_a), 'fdr_compXlineage': int(g_b)})

# C5
for tissue_label in ['Liver', 'Blood']:
    fpath = os.path.join(OUT_BASE, 'Fix1B_StratifiedFDR', f'C5_{tissue_label}_stratified_FDR.csv')
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        p_col = [c for c in df.columns if 'p_val' in c.lower() or c == 'p_value'][0]
        nom = (df[p_col] < 0.05).sum()
        g_a = df['fdr_sig_compOnly'].sum() if 'fdr_sig_compOnly' in df.columns else 0
        g_b = df['fdr_sig_compXlineage'].sum() if 'fdr_sig_compXlineage' in df.columns else 0
        print(f"  C5_{tissue_label:<15} {nom:<10} {'—':<10} {int(g_a):<15} {int(g_b):<18}")
        summary_rows.append({'dataset': f'C5_{tissue_label}', 'nominal': nom, 'fdr_compOnly': int(g_a), 'fdr_compXlineage': int(g_b)})

# C7
print(f"  C7{'_All':<16} {nominal:<10} {surv_g:<10} {'—':<15} {surv_s:<18}")

print()
print("NOTE: If Level B (comp×tissue×lineage) still yields 0 survivors,")
print("  the manuscript should report nominal p-values with effect sizes")
print("  and acknowledge the FDR limitation due to small donor n.")
print("  C7 correlations are robust (57.6%+ global FDR survival).")

# Save summary
pd.DataFrame(summary_rows).to_csv(
    os.path.join(OUT_BASE, 'Fix1B_StratifiedFDR', 'FDR_stratified_summary.csv'), index=False)
print(f"\nSaved summary to Fix1B_StratifiedFDR/FDR_stratified_summary.csv")


  FIX 1B: STRATIFIED FDR SUMMARY

Dataset              Nominal    Global     Comp×Tissue     Comp×Tissue×Lin    Best Strategy
-----------------------------------------------------------------------------------------------
  C3_Liver           583        —          0               0                  nominal+ES
  C3_Blood           859        —          0               53                 B
  C4_Liver           108        —          0               0                 
  C4_Blood           130        —          0               20                
  C5_Liver           398        —          0               0                 
  C5_Blood           610        —          0               0                 
  C7_All             517        308        —               333               

NOTE: If Level B (comp×tissue×lineage) still yields 0 survivors,
  the manuscript should report nominal p-values with effect sizes
  and acknowledge the FDR limitation due to small donor n.
  C7 correlations are robust

## Cell 8. Fix 3B — C3-Only Gene Rerun (Load Data)

**20 critical C3-only genes** to be analyzed in C5-style format:

| Category | Genes |
|----------|-------|
| IFN response | MX1, ISG15, STAT2, IRF3, IRF7 |
| JAK-STAT brakes | SOCS1, SOCS3 |
| B cell | AICDA, JCHAIN |
| Antigen presentation | HLA-DRA, HLA-DRB1, HLA-DPB1, HLA-DPA1, CD74, B2M, TAP1 |
| Tolerance/Other | IL1RN, STAT1 |
| Inflammasome | IL1B |


In [11]:
# ================================================================
# CELL 8: FIX 3B — LOAD h5ad + DEFINE C3-ONLY GENES
# ================================================================
import scanpy as sc
import numpy as np

print("Loading h5ad...")
adata = sc.read_h5ad(DATA_PATH)
print(f"  adata: {adata.shape[0]} cells x {adata.shape[1]} genes")

# --- Robust Column Detection ---
print("Detecting columns...")
obs_keys = adata.obs.columns

# Helper to find column
def find_col(candidates, keys):
    for c in candidates:
        if c in keys: return c
    return None

# 1. Tissue
tissue_col = find_col(['tissue', 'Tissue', 'tissue_source'], obs_keys)
if not tissue_col:
    print("⚠️ Tissue column not found, using default 'tissue'")
    tissue_col = 'tissue'

# 2. Lineage
lineage_col = find_col(['major_lineage', 'lineage', 'Lineage', 'cell_type'], obs_keys)
if not lineage_col:
    print("⚠️ Lineage column not found, using default 'major_lineage'")
    lineage_col = 'major_lineage'

# 3. Stage
stage_col = find_col(['Stage', 'stage', 'Group', 'group', 'condition'], obs_keys)
if not stage_col:
    print("⚠️ Stage column not found, using default 'Stage'")
    stage_col = 'Stage'

# 4. Donor / Sample
# Prioritize 'donor' (biological) -> 'sample' (technical) -> others
donor_col = find_col(['donor', 'Donor', 'sample', 'Sample', 'subject', 'patient', 'participant_id'], obs_keys)
if not donor_col:
    print("⚠️ Donor/Sample column not found, using default 'sample'")
    donor_col = 'sample'

print(f"  tissue:  {tissue_col}")
print(f"  lineage: {lineage_col}")
print(f"  stage:   {stage_col}")
if donor_col in adata.obs:
    print(f"  donor:   {donor_col} ({adata.obs[donor_col].nunique()} unique)")
else:
    print(f"  donor:   {donor_col} (Column missing in adata.obs)")

# 20 critical C3-only genes
C3_ONLY_GENES = [
    'MX1', 'ISG15', 'STAT2', 'IRF3', 'IRF7',
    'SOCS1', 'SOCS3',
    'AICDA', 'JCHAIN',
    'HLA-DRA', 'HLA-DRB1', 'HLA-DPB1', 'HLA-DPA1', 'CD74', 'B2M', 'TAP1',
    'IL1RN', 'STAT1',
    'IL1B',
]

# Check availability
available = [g for g in C3_ONLY_GENES if g in adata.var_names]
missing = [g for g in C3_ONLY_GENES if g not in adata.var_names]
print(f"\n  Available: {len(available)}/{len(C3_ONLY_GENES)}")
if missing:
    print(f"  Missing: {missing}")

# Define lineages
actual_lineages = adata.obs[lineage_col].unique().tolist() if lineage_col in adata.obs else []
LINEAGES = ['Myeloid', 'CD4_T', 'CD8_T', 'NK', 'B', 'PlasmaB']
# Adjust based on data
found_lineages = [l for l in LINEAGES if l in actual_lineages]
if found_lineages:
    LINEAGES = found_lineages
else:
    print(f"  WARNING: Standard lineages not found. Available: {actual_lineages[:5]}...")

print(f"  Lineages to analyze: {LINEAGES}")

# Define comparisons
COMPARISONS = [
    ('NL', 'IT'), ('NL', 'IA'), ('NL', 'AR'), ('NL', 'CR'),
    ('IT', 'IA'), ('IA', 'AR'), ('IT', 'CR'),
]

# Stage mapping
STAGE_MAP = {}
if stage_col in adata.obs:
    actual_stages = adata.obs[stage_col].unique().tolist()
    for s in actual_stages:
        s_str = str(s)
        if 'Normal' in s_str or s_str == 'NL' or 'HC' in s_str: STAGE_MAP['NL'] = s
        elif 'Tolerant' in s_str or s_str == 'IT': STAGE_MAP['IT'] = s
        elif 'Active' in s_str or s_str == 'IA': STAGE_MAP['IA'] = s
        elif 'Resolved' in s_str and ('Acute' in s_str or 'AR' in s_str.upper()): STAGE_MAP['AR'] = s
        elif ('Chronic' in s_str and 'Resolved' in s_str) or s_str == 'CR': STAGE_MAP['CR'] = s
        elif s_str in ['NL', 'IT', 'IA', 'AR', 'CR']: STAGE_MAP[s_str] = s

print(f"  Stage mapping: {STAGE_MAP}")
print("\nReady for C5-style analysis")

Loading h5ad...
  adata: 243000 cells x 24452 genes
Detecting columns...
  tissue:  tissue
  lineage: major_lineage
  stage:   Stage
  donor:   sample (46 unique)

  Available: 19/19
  Lineages to analyze: ['Myeloid', 'CD4_T', 'CD8_T', 'NK', 'B', 'PlasmaB']
  Stage mapping: {'IT': 'IT', 'AR': 'AR', 'IA': 'IA', 'NL': 'NL', 'CR': 'CR'}

Ready for C5-style analysis


## Cell 9. Fix 3B — Run C5-Style Mann-Whitney U for 20 Genes

In [12]:
# ================================================================
# CELL 9: FIX 3B — C5-STYLE ANALYSIS FOR C3-ONLY GENES
# ================================================================
import scipy.sparse as sp
from scipy.stats import mannwhitneyu

print("=" * 60)
print("  FIX 3B: C5-STYLE ANALYSIS FOR 20 C3-ONLY GENES")
print("=" * 60)

results = []
total = len(available) * len(LINEAGES) * len(COMPARISONS) * 2  # x2 for tissues
done = 0

TISSUES = adata.obs[tissue_col].unique().tolist()

for tissue in TISSUES:
    tissue_mask = adata.obs[tissue_col] == tissue
    for lineage in LINEAGES:
        lineage_mask = adata.obs[lineage_col] == lineage
        combined_mask = tissue_mask & lineage_mask

        for grp1_abbr, grp2_abbr in COMPARISONS:
            if grp1_abbr not in STAGE_MAP or grp2_abbr not in STAGE_MAP:
                continue

            grp1_name = STAGE_MAP[grp1_abbr]
            grp2_name = STAGE_MAP[grp2_abbr]

            mask1 = combined_mask & (adata.obs[stage_col] == grp1_name)
            mask2 = combined_mask & (adata.obs[stage_col] == grp2_name)

            # Get donor-level means
            donors1 = adata.obs.loc[mask1, donor_col].unique()
            donors2 = adata.obs.loc[mask2, donor_col].unique()

            if len(donors1) < 2 or len(donors2) < 2:
                done += len(available)
                continue

            for gene in available:
                done += 1
                if gene not in adata.var_names:
                    continue

                gene_idx = list(adata.var_names).index(gene)

                # Donor-level mean expression
                vals1 = []
                for d in donors1:
                    d_mask = mask1 & (adata.obs[donor_col] == d)
                    cells = adata[d_mask]
                    if cells.shape[0] == 0:
                        vals1.append(0.0)
                        continue
                    X = cells.X[:, gene_idx]
                    if sp.issparse(X):
                        X = X.toarray().flatten()
                    vals1.append(float(np.mean(X)))

                vals2 = []
                for d in donors2:
                    d_mask = mask2 & (adata.obs[donor_col] == d)
                    cells = adata[d_mask]
                    if cells.shape[0] == 0:
                        vals2.append(0.0)
                        continue
                    X = cells.X[:, gene_idx]
                    if sp.issparse(X):
                        X = X.toarray().flatten()
                    vals2.append(float(np.mean(X)))

                vals1 = np.array(vals1)
                vals2 = np.array(vals2)

                mean1 = np.mean(vals1)
                mean2 = np.mean(vals2)
                fc_pct = ((mean2 - mean1) / mean1 * 100) if mean1 > 0 else (99999.0 if mean2 > 0 else 0.0)

                # Mann-Whitney U
                try:
                    stat, pval = mannwhitneyu(vals1, vals2, alternative='two-sided')
                except:
                    pval = 1.0
                    stat = 0.0

                # Consistency count
                n_pairs = len(vals1) * len(vals2)
                concordant = sum(1 for v1 in vals1 for v2 in vals2 if v2 > v1)
                consistency = f"{concordant}/{n_pairs}"

                results.append({
                    'tissue': tissue,
                    'lineage': lineage,
                    'gene': gene,
                    'comparison': f'{grp1_abbr}_vs_{grp2_abbr}',
                    'grp1': grp1_abbr,
                    'grp2': grp2_abbr,
                    'n1': len(donors1),
                    'n2': len(donors2),
                    'mean_grp1': mean1,
                    'mean_grp2': mean2,
                    'fold_change_pct': fc_pct,
                    'U_stat': stat,
                    'p_value': pval,
                    'consistency': consistency,
                    'significant': pval < 0.05,
                })

                if done % 500 == 0:
                    print(f"  {done}/{total} ({done/total*100:.0f}%)")

df_results = pd.DataFrame(results)
print(f"\n  Total results: {df_results.shape[0]}")
print(f"  Significant (nominal p<0.05): {df_results['significant'].sum()}")

# Apply stratified FDR (comparison × tissue × lineage)
df_results = stratified_fdr(df_results, 'p_value', ['comparison', 'tissue', 'lineage'], label='compXtissueXlineage')
# Also comparison × tissue
df_results = stratified_fdr(df_results, 'p_value', ['comparison', 'tissue'], label='compXtissue')

surv_b = (df_results['fdr_sig_compXtissueXlineage'] == True).sum()
surv_a = (df_results['fdr_sig_compXtissue'] == True).sum()
print(f"  FDR survivors (comp×tissue): {surv_a}")
print(f"  FDR survivors (comp×tissue×lineage): {surv_b}")

# Save
out_path = os.path.join(OUT_BASE, 'Fix3B_C3onlyGenes', 'C3only_20genes_C5style_results.csv')
df_results.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

  FIX 3B: C5-STYLE ANALYSIS FOR 20 C3-ONLY GENES
  500/1596 (31%)
  1000/1596 (63%)
  1500/1596 (94%)

  Total results: 1596
  Significant (nominal p<0.05): 212
  FDR survivors (comp×tissue): 40
  FDR survivors (comp×tissue×lineage): 51

Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis/C9_method_fixes/Fix3B_C3onlyGenes/C3only_20genes_C5style_results.csv


## Cell 10. Fix 3B — NL→IT Key Results for 20 C3-Only Genes

In [13]:
# ================================================================
# CELL 10: FIX 3B — KEY RESULTS REPORT
# ================================================================
print("=" * 60)
print("  FIX 3B: NL→IT KEY RESULTS (20 C3-ONLY GENES)")
print("=" * 60)

# Filter NL→IT
mask_it = df_results['comparison'].str.contains('NL') & df_results['comparison'].str.contains('IT')
df_it = df_results[mask_it].copy()

# Significant findings
df_sig = df_it[df_it['significant']].sort_values('p_value')
print(f"\n  NL→IT significant (nominal): {len(df_sig)} / {len(df_it)}")

# Group by gene category
ifn_genes = ['MX1', 'ISG15', 'STAT2', 'IRF3', 'IRF7']
socs_genes = ['SOCS1', 'SOCS3']
ag_genes = ['HLA-DRA', 'HLA-DRB1', 'HLA-DPB1', 'HLA-DPA1', 'CD74', 'B2M', 'TAP1']
bcell_genes = ['AICDA', 'JCHAIN']
other_genes = ['IL1RN', 'STAT1', 'IL1B']

categories = [
    ('IFN Response', ifn_genes),
    ('JAK-STAT Brakes', socs_genes),
    ('Antigen Presentation', ag_genes),
    ('B Cell', bcell_genes),
    ('Other', other_genes),
]

for cat_name, genes in categories:
    print(f"\n── {cat_name} ──")
    for gene in genes:
        for tissue in TISSUES:
            for lineage in LINEAGES:
                row = df_it[(df_it['gene'] == gene) & (df_it['tissue'] == tissue) & (df_it['lineage'] == lineage)]
                if row.empty:
                    continue
                r = row.iloc[0]
                sig_mark = "★" if r['significant'] else " "
                fdr_mark = ""
                if 'fdr_sig_compXtissueXlineage' in r.index and r['fdr_sig_compXtissueXlineage']:
                    fdr_mark = " [FDR★]"
                direction = "↑" if r['fold_change_pct'] > 0 else "↓"
                if r['significant']:
                    print(f"  {sig_mark} {gene:<12} {tissue:<6} {lineage:<10} {direction}{abs(r['fold_change_pct']):.1f}% p={r['p_value']:.4f} ({r['consistency']}){fdr_mark}")

# ── IT-Specific Check ──
print()
print("=" * 60)
print("  IT-SPECIFIC CLASSIFICATION (NL→IT sig, NL→IA NS)")
print("=" * 60)

mask_ia = df_results['comparison'].str.contains('NL') & df_results['comparison'].str.contains('IA')
df_ia = df_results[mask_ia].copy()

it_specific = []
for _, row_it in df_sig.iterrows():
    gene = row_it['gene']
    tissue = row_it['tissue']
    lineage = row_it['lineage']
    # Find matching NL→IA
    match = df_ia[(df_ia['gene'] == gene) & (df_ia['tissue'] == tissue) & (df_ia['lineage'] == lineage)]
    if not match.empty:
        p_ia = match.iloc[0]['p_value']
        if p_ia >= 0.05:  # NL→IA NS → IT-specific
            it_specific.append({
                'gene': gene, 'tissue': tissue, 'lineage': lineage,
                'IT_pct': row_it['fold_change_pct'], 'IT_p': row_it['p_value'],
                'IA_p': p_ia, 'consistency': row_it['consistency']
            })

if it_specific:
    df_itspec = pd.DataFrame(it_specific)
    print(f"\n  IT-specific gene-lineage combos: {len(df_itspec)}")
    for _, r in df_itspec.iterrows():
        direction = "↑" if r['IT_pct'] > 0 else "↓"
        print(f"  ★ {r['gene']:<12} {r['tissue']:<6} {r['lineage']:<10} {direction}{abs(r['IT_pct']):.1f}% IT_p={r['IT_p']:.4f} IA_p={r['IA_p']:.3f}")

    out_path = os.path.join(OUT_BASE, 'Fix3B_C3onlyGenes', 'C3only_IT_specific_genes.csv')
    df_itspec.to_csv(out_path, index=False)
    print(f"\n  Saved: {out_path}")
else:
    print("  No IT-specific genes found (all NL→IA also significant)")


  FIX 3B: NL→IT KEY RESULTS (20 C3-ONLY GENES)

  NL→IT significant (nominal): 62 / 228

── IFN Response ──
  ★ MX1          Blood  Myeloid    ↑319.2% p=0.0025 (35/35) [FDR★]
  ★ MX1          Blood  CD4_T      ↑66.0% p=0.0480 (30/35)
  ★ MX1          Blood  PlasmaB    ↑166.2% p=0.0147 (33/35)
  ★ MX1          Liver  Myeloid    ↑133.9% p=0.0411 (31/36)
  ★ MX1          Liver  CD8_T      ↑85.0% p=0.0152 (33/36)
  ★ MX1          Liver  B          ↑324.7% p=0.0043 (30/30)
  ★ ISG15        Blood  Myeloid    ↑397.1% p=0.0025 (35/35) [FDR★]
  ★ ISG15        Blood  PlasmaB    ↑346.3% p=0.0335 (31/35)
  ★ STAT2        Blood  Myeloid    ↑191.5% p=0.0025 (35/35) [FDR★]
  ★ STAT2        Blood  B          ↑132.2% p=0.0051 (34/35) [FDR★]
  ★ STAT2        Liver  CD4_T      ↑181.3% p=0.0087 (34/36)
  ★ IRF3         Blood  Myeloid    ↑102.0% p=0.0025 (35/35) [FDR★]
  ★ IRF3         Blood  CD4_T      ↑67.5% p=0.0303 (31/35)
  ★ IRF3         Blood  B          ↑94.8% p=0.0051 (34/35) [FDR★]
  ★ IRF3      

## Cell 11. Fix 3B — Donor-Level Dot Plots for Key C3-Only Genes

In [14]:
# ================================================================
# CELL 11: FIX 3B — DOT PLOTS (KEY GENES)
# ================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import scipy.sparse as sp

print("Generating dot plots for key C3-only genes...")

# Key genes to plot (most important for manuscript)
PLOT_GENES = ['MX1', 'ISG15', 'SOCS1', 'SOCS3', 'AICDA', 'HLA-DRA', 'HLA-DPB1', 'IL1RN']
PLOT_GENES = [g for g in PLOT_GENES if g in available]

STAGES_ORDER = ['NL', 'IT', 'IA', 'AR', 'CR']

for gene in PLOT_GENES:
    fig, axes = plt.subplots(2, len(LINEAGES), figsize=(3.5*len(LINEAGES), 8),
                              constrained_layout=True)
    fig.suptitle(f'{gene} — Donor-Level Expression', fontsize=14, fontweight='bold')

    gene_idx = list(adata.var_names).index(gene)

    for t_idx, tissue in enumerate(TISSUES):
        tissue_mask = adata.obs[tissue_col] == tissue
        color = '#CC3333' if 'liver' in tissue.lower() or 'Liver' in tissue else '#3366CC'
        marker = 'o' if 'liver' in tissue.lower() or 'Liver' in tissue else '^'

        for l_idx, lineage in enumerate(LINEAGES):
            ax = axes[t_idx, l_idx] if len(TISSUES) > 1 else axes[l_idx]
            lineage_mask = adata.obs[lineage_col] == lineage

            for s_idx, stage_abbr in enumerate(STAGES_ORDER):
                if stage_abbr not in STAGE_MAP:
                    continue
                stage_name = STAGE_MAP[stage_abbr]
                mask = tissue_mask & lineage_mask & (adata.obs[stage_col] == stage_name)
                donors = adata.obs.loc[mask, donor_col].unique()

                donor_means = []
                for d in donors:
                    d_mask = mask & (adata.obs[donor_col] == d)
                    cells = adata[d_mask]
                    if cells.shape[0] == 0:
                        donor_means.append(0.0)
                        continue
                    X = cells.X[:, gene_idx]
                    if sp.issparse(X):
                        X = X.toarray().flatten()
                    donor_means.append(float(np.mean(X)))

                x_jitter = s_idx + np.random.normal(0, 0.05, len(donor_means))
                ax.scatter(x_jitter, donor_means, c=color, marker=marker,
                          s=40, alpha=0.7, edgecolors='black', linewidths=0.5)

                if donor_means:
                    ax.plot([s_idx-0.15, s_idx+0.15], [np.mean(donor_means)]*2,
                           color=color, linewidth=2)

            ax.set_xticks(range(len(STAGES_ORDER)))
            ax.set_xticklabels(STAGES_ORDER, fontsize=9)
            ax.set_title(f'{lineage}', fontsize=10)
            if l_idx == 0:
                ax.set_ylabel(f'{tissue}\nMean Expression', fontsize=9)

    fig.savefig(os.path.join(OUT_BASE, 'Fix3B_C3onlyGenes', 'dotplots', f'{gene}_dotplot.png'),
                dpi=300, facecolor='white', bbox_inches='tight')
    plt.close(fig)
    print(f"  ✅ {gene}")

print(f"\nAll dot plots saved to Fix3B_C3onlyGenes/dotplots/")


Generating dot plots for key C3-only genes...
  ✅ MX1
  ✅ ISG15
  ✅ SOCS1
  ✅ SOCS3
  ✅ AICDA
  ✅ HLA-DRA
  ✅ HLA-DPB1
  ✅ IL1RN

All dot plots saved to Fix3B_C3onlyGenes/dotplots/


## Cell 12. Final Summary — C9B Complete

In [15]:
# ================================================================
# CELL 12: FINAL SUMMARY
# ================================================================
print("=" * 60)
print("  C9B COMPLETE — FINAL SUMMARY")
print("=" * 60)

print()
print("[FIX 1B] STRATIFIED FDR")
print("  Three stratification levels applied to C3/C4/C5/C7")
print("  Level A: comparison × tissue")
print("  Level B: comparison × tissue × lineage (finest)")
print("  See Fix1B_StratifiedFDR/ for all CSVs with q-values")
print()

print("[FIX 3B] C3-ONLY GENES RERUN")
print(f"  {len(available)} genes analyzed in C5-style format")
print("  Stratified FDR applied to new results")
print("  IT-specific classification completed")
print("  Dot plots generated for key genes")
print("  See Fix3B_C3onlyGenes/ for all outputs")
print()

print("ALL OUTPUTS:")
print(f"  {OUT_BASE}/")
print(f"  ├── Fix1B_StratifiedFDR/")
print(f"  │   ├── C3_stratified_FDR.csv")
print(f"  │   ├── C4_Liver_stratified_FDR.csv")
print(f"  │   ├── C4_Blood_stratified_FDR.csv")
print(f"  │   ├── C5_Liver_stratified_FDR.csv")
print(f"  │   ├── C5_Blood_stratified_FDR.csv")
print(f"  │   ├── C7_stratified_FDR.csv")
print(f"  │   └── FDR_stratified_summary.csv")
print(f"  └── Fix3B_C3onlyGenes/")
print(f"      ├── C3only_20genes_C5style_results.csv")
print(f"      ├── C3only_IT_specific_genes.csv")
print(f"      └── dotplots/  (8 gene dot plots)")
print()
print("NEXT STEPS:")
print("  1. Review FDR stratified summary → decide reporting threshold")
print("  2. If Level B still 0%: report nominal p + effect size + consistency")
print("  3. Integrate C3-only gene results into Results Draft")
print("  4. Update Methods: 'Two gene panels analyzed (C3: 196, C5: 148)'")
print("  5. Update AICDA claim: Blood AICDA (6/7 IT donors) replaces Liver claim")
print()
print("✅ C9B COMPLETE")


  C9B COMPLETE — FINAL SUMMARY

[FIX 1B] STRATIFIED FDR
  Three stratification levels applied to C3/C4/C5/C7
  Level A: comparison × tissue
  Level B: comparison × tissue × lineage (finest)
  See Fix1B_StratifiedFDR/ for all CSVs with q-values

[FIX 3B] C3-ONLY GENES RERUN
  19 genes analyzed in C5-style format
  Stratified FDR applied to new results
  IT-specific classification completed
  Dot plots generated for key genes
  See Fix3B_C3onlyGenes/ for all outputs

ALL OUTPUTS:
  /content/drive/MyDrive/ITLAS/results/version18-analysis/C9_method_fixes/
  ├── Fix1B_StratifiedFDR/
  │   ├── C3_stratified_FDR.csv
  │   ├── C4_Liver_stratified_FDR.csv
  │   ├── C4_Blood_stratified_FDR.csv
  │   ├── C5_Liver_stratified_FDR.csv
  │   ├── C5_Blood_stratified_FDR.csv
  │   ├── C7_stratified_FDR.csv
  │   └── FDR_stratified_summary.csv
  └── Fix3B_C3onlyGenes/
      ├── C3only_20genes_C5style_results.csv
      ├── C3only_IT_specific_genes.csv
      └── dotplots/  (8 gene dot plots)

NEXT STEPS:
